# Fine-Tuning DistilBERT for Scam Message Detection (Google Colab)

This notebook provides a complete, self-contained pipeline to fine-tune **DistilBERT** (or `xlm-roberta-base`) on Google Colab with GPU acceleration, comparing transformer contextual embeddings against our traditional ML baseline.

### Instructions for Running in Google Colab:
1. Open [Google Colab](https://colab.research.google.com) and upload this notebook.
2. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
3. Run the cells sequentially.

In [ ]:
# Cell 1: Environment Setup & Dependencies
!pip install -q transformers datasets accelerate evaluate scikit-learn torch

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Import Libraries & Dataset Ingestion
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# If running in Colab, you can either upload train.csv/test.csv or fetch the UCI dataset directly:
import urllib.request, zipfile, os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
urllib.request.urlretrieve(url, "sms.zip")
with zipfile.ZipFile("sms.zip", "r") as zip_ref:
    zip_ref.extractall("data")

df = pd.read_csv("data/SMSSpamCollection", sep="\t", names=["raw_label", "text"], quoting=3, encoding="utf-8", on_bad_lines="skip")
df["label"] = df["raw_label"].map({"ham": 0, "spam": 1})
df = df.dropna().drop_duplicates(subset=["text"]).reset_index(drop=True)

from sklearn.model_selection import train_test_split
train_val, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["label"])
train_df, val_df = train_test_split(train_val, test_size=0.1765, random_state=42, stratify=train_val["label"])

print(f"Splits -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
# Cell 3: Tokenization using DistilBERT Tokenizer
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

ds_train = Dataset.from_pandas(train_df[["text", "label"]])
ds_val = Dataset.from_pandas(val_df[["text", "label"]])
ds_test = Dataset.from_pandas(test_df[["text", "label"]])

dataset = DatasetDict({"train": ds_train, "validation": ds_val, "test": ds_test})
tokenized_dataset = dataset.map(tokenize_fn, batched=True)

In [ ]:
# Cell 4: Initialize Pretrained Model with Sequence Classification Head
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision = evaluate.load("precision").compute(predictions=predictions, references=labels, pos_label=1)["precision"]
    recall = evaluate.load("recall").compute(predictions=predictions, references=labels, pos_label=1)["recall"]
    f1 = evaluate.load("f1").compute(predictions=predictions, references=labels, pos_label=1)["f1"]
    return {"scam_precision": precision, "scam_recall": recall, "scam_f1": f1}

In [ ]:
# Cell 5: Training Setup & Execution
training_args = TrainingArguments(
    output_dir="./distilbert-scam-detector",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="scam_f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 6: Final Evaluation on Unseen Test Split
test_predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = np.argmax(test_predictions.predictions, axis=-1)
true_labels = test_predictions.label_ids

print("\n--- Transformer (DistilBERT) Test Evaluation ---")
print(classification_report(true_labels, pred_labels, target_names=["Legitimate", "Scam"]))
print("Confusion Matrix:")
print(confusion_matrix(true_labels, pred_labels))